# LeetCode 729 — My Calendar I bằng Python

Link đề: https://leetcode.com/problems/my-calendar-i/ · Độ khó: **Medium** · Chủ đề: Design, Segment Tree, Array, Binary Search, Ordered Set

Skeleton LeetCode:

```python
class MyCalendar:

    def __init__(self):
        ...

    def book(self, startTime: int, endTime: int) -> bool:
        ...


# Your MyCalendar object will be instantiated and called as such:
# obj = MyCalendar()
# param_1 = obj.book(startTime,endTime)
```

Ghi chú: đây là bài **thiết kế lớp** (class design). Cốt lõi là một câu hỏi: "khoảng `[start, end)` mới có **giao** với khoảng nào đã đặt không?" và cách lưu các khoảng sao cho trả lời nhanh. Python không có sẵn "ordered set" như `TreeMap` của Java, nên ta dùng `bisect` trên list đã sắp xếp (chỉ stdlib). Notebook cũng tự dựng một **bộ chạy chuỗi lệnh** mô phỏng cách LeetCode gọi lớp.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đặt lịch theo khoảng **nửa mở** `[start, end)`; chỉ nhận lịch mới nếu **không trùng** (không có thời điểm chung) với lịch đã nhận.
2. Nắm các công cụ Python: lớp với `__init__`/phương thức, `tuple`, `any`, `bisect_left`/`bisect_right`, `list.insert`, cấu trúc nút cây (BST, segment tree động).
3. Đi từ **quét mọi lịch cũ** → **danh sách đã sắp xếp + kiểm tra hàng xóm** → **sắp theo `end` chỉ cần kiểm tra một ứng viên** → **cây nhị phân tìm kiếm** → **segment tree động**.
4. Hiểu mấu chốt: hai khoảng nửa mở `[a, b)` và `[c, d)` giao nhau **khi và chỉ khi `a < d` và `c < b`**; trong tập khoảng rời nhau đã sắp xếp, chỉ **một ứng viên** có thể giao với khoảng mới.
5. Tránh các lỗi kinh điển: dùng `<=` (coi hai lịch sát nhau là trùng), chỉ kiểm tra điểm đầu của lịch mới (bỏ sót lịch mới **bao** lịch cũ), quên lưu lịch khi thành công, nhầm `bisect_left`/`bisect_right`.

## 2. Đề bài & ví dụ

Bạn đang viết chương trình lịch. Ta có thể thêm một sự kiện mới nếu việc thêm **không gây đặt trùng** (double booking). Đặt trùng xảy ra khi hai sự kiện có **giao khác rỗng** (có một thời điểm chung).

Sự kiện là cặp số nguyên `startTime`, `endTime`, biểu diễn khoảng **nửa mở** `[startTime, endTime)`: các số thực `x` thỏa `startTime <= x < endTime`.

Cài đặt lớp `MyCalendar`:

- `MyCalendar()`: khởi tạo lịch.
- `boolean book(int startTime, int endTime)`: trả `true` nếu thêm được sự kiện mà không gây đặt trùng; ngược lại trả `false` và **không** thêm sự kiện.

**Ví dụ 1**

```
Input:  ["MyCalendar", "book", "book", "book"]
        [[], [10, 20], [15, 25], [20, 30]]
Output: [null, true, false, true]
```

Giải thích: `book(10, 20)` thành công. `book(15, 25)` thất bại vì thời điểm 15 đã bị chiếm. `book(20, 30)` thành công vì sự kiện đầu chỉ chiếm các thời điểm **nhỏ hơn 20** (không gồm 20).

**Ràng buộc:**

- `0 <= start < end <= 10^9`
- Tối đa `1000` lần gọi `book`.

Hệ quả: tọa độ rất lớn (tới `10^9`) nên **không** thể đánh dấu từng thời điểm bằng mảng; nhưng chỉ có tối đa 1000 sự kiện nên danh sách các khoảng luôn nhỏ.

In [1]:
from bisect import bisect_left, bisect_right
import random
import time

example_ops = ["MyCalendar", "book", "book", "book"]
example_args = [[], [10, 20], [15, 25], [20, 30]]
example_out = [None, True, False, True]


def run_design(cls, ops, args):
    """Bộ chạy chuỗi lệnh mô phỏng LeetCode: ops[0] là tên lớp (hàm khởi tạo), phần còn lại là tên phương thức."""
    obj = None
    out = []
    for op, a in zip(ops, args):
        if op == cls.__name__ or (obj is None and op == "MyCalendar"):
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out


print("ví dụ đề (ops):", example_ops)
print("args          :", example_args)
print("kỳ vọng       :", example_out)

ví dụ đề (ops): ['MyCalendar', 'book', 'book', 'book']
args          : [[], [10, 20], [15, 25], [20, 30]]
kỳ vọng       : [None, True, False, True]


## 3. Kiến thức nền

### 3.1. Khoảng nửa mở `[start, end)`

`[10, 20)` gồm mọi thời điểm `x` với `10 <= x < 20`: **gồm 10, không gồm 20**. Nhờ vậy hai sự kiện "sát nhau" như `[10, 20)` và `[20, 30)` **không** chung thời điểm nào: thời điểm 20 chỉ thuộc sự kiện thứ hai.

Với số nguyên, `[10, 20)` ứng với các số `10, 11, ..., 19` (và mọi số thực giữa chúng).

In [2]:
def points(start, end):
    """Tập các số nguyên của khoảng [start, end) (chỉ dùng cho khoảng nhỏ)."""
    return set(range(start, end))


a = points(10, 20)
b = points(20, 30)
print(sorted(a)[:3], "...", sorted(a)[-1])
print("hai khoảng sát nhau có chung điểm nào không?", bool(a & b))
print("[10,20) và [15,25) chung:", sorted(points(10, 20) & points(15, 25)))

[10, 11, 12] ... 19
hai khoảng sát nhau có chung điểm nào không? False
[10,20) và [15,25) chung: [15, 16, 17, 18, 19]


### 3.2. Điều kiện hai khoảng giao nhau

Hai khoảng `[a, b)` và `[c, d)` **không giao** khi cái này kết thúc **trước hoặc đúng lúc** cái kia bắt đầu: `b <= c` (khoảng thứ nhất nằm hoàn toàn bên trái) hoặc `d <= a` (khoảng thứ hai nằm hoàn toàn bên trái). Phủ định của "`b <= c` hoặc `d <= a`" là:

```
giao nhau  ⇔  a < d  và  c < b
```

Đây là công thức quan trọng nhất của bài. Cell dưới kiểm chứng công thức bằng cách so với phép giao tập điểm.

In [3]:
def overlap(a, b, c, d):
    return a < d and c < b


# so với định nghĩa bằng tập điểm, mọi cặp khoảng nhỏ trên [0, 8]
for a in range(0, 8):
    for b in range(a + 1, 9):
        for c in range(0, 8):
            for d in range(c + 1, 9):
                assert overlap(a, b, c, d) == bool(points(a, b) & points(c, d)), (a, b, c, d)
print("công thức a < d và c < b khớp định nghĩa tập điểm")
print(overlap(10, 20, 20, 30), overlap(10, 20, 15, 25), overlap(10, 20, 12, 15), overlap(12, 15, 10, 20))

công thức a < d và c < b khớp định nghĩa tập điểm
False True True True


### 3.3. Lớp (class) và bộ chạy chuỗi lệnh

Bài thiết kế lớp cho một dãy lệnh như `["MyCalendar", "book", "book"]` với đối số `[[], [10, 20], [15, 25]]`. Lệnh đầu gọi hàm khởi tạo `__init__`; các lệnh sau gọi phương thức cùng tên trên **cùng một đối tượng** (nên trạng thái được giữ lại giữa các lần gọi). Hàm `run_design` ở mục 2 thực hiện đúng việc đó: `getattr(obj, "book")(*args)`.

In [4]:
class Counter:
    def __init__(self):
        self.n = 0

    def inc(self, d):
        self.n += d
        return self.n


print(run_design(Counter, ["Counter", "inc", "inc", "inc"], [[], [2], [3], [10]]))   # [None, 2, 5, 15]

[None, 2, 5, 15]


### 3.4. `bisect_left`, `bisect_right`, `insert`

Với list `a` đã **sắp xếp tăng**:

- `bisect_left(a, x)`: vị trí `i` nhỏ nhất sao cho các phần tử trước `i` đều `< x` (chèn `x` vào đây thì nằm **trước** các số bằng `x`).
- `bisect_right(a, x)`: vị trí `i` sao cho các phần tử trước `i` đều `<= x` (nằm **sau** các số bằng `x`).
- `a.insert(i, x)`: chèn `x` vào vị trí `i` (dịch phần sau sang phải; `O(len)` nhưng rất nhanh vì là `memmove`).
- Tìm kiếm là `O(log n)`.

In [5]:
a = [10, 20, 20, 30]
for x in (5, 20, 25, 99):
    print(f"x={x:2d}: bisect_left={bisect_left(a, x)}, bisect_right={bisect_right(a, x)}")
a.insert(bisect_right(a, 25), 25)
print(a)

x= 5: bisect_left=0, bisect_right=0
x=20: bisect_left=1, bisect_right=3
x=25: bisect_left=3, bisect_right=3
x=99: bisect_left=4, bisect_right=4
[10, 20, 20, 25, 30]


### 3.5. Cây nhị phân tìm kiếm (BST) và segment tree — đọc nhanh

- **BST**: mỗi nút lưu một thứ; con trái "nhỏ hơn", con phải "lớn hơn". Tra cứu/chèn đi từ gốc xuống một nhánh: `O(chiều cao)`. Chiều cao ~`log n` khi dữ liệu ngẫu nhiên, nhưng tệ nhất là `n` nếu chèn theo thứ tự đã sắp xếp.
- **Segment tree động**: cây chia đôi **đoạn tọa độ** `[0, 2^30)`; mỗi nút phụ trách một đoạn con. Ta chỉ tạo nút khi cần, nên bộ nhớ chỉ `O(số sự kiện * 30)`. Dùng để hỏi "có chỗ nào trong `[l, r)` đã bị đặt chưa?" và đánh dấu một khoảng đã đặt.

Hai cách này sẽ được dựng ở mục 7.

## 4. Ý tưởng nền tảng / trực giác

Làm tay với ví dụ 1. Vẽ trục thời gian:

```
thời gian:   10        15   20   25   30
book(10,20): [==========)
book(15,25):      [===========)           <- chồng lên [15, 20) => false
book(20,30):                [===========) <- chỉ sát nhau tại 20 => true
```

Cách tự nhiên nhất: giữ **danh sách mọi sự kiện đã nhận**. Mỗi lần có sự kiện mới `[start, end)`, **xem lần lượt từng sự kiện cũ** `[s, e)`; nếu `start < e` và `s < end` thì trùng → trả `False`. Hết danh sách mà không thấy trùng → thêm vào, trả `True`.

Ví dụ: sau `book(10, 20)`, kiểm `book(15, 25)`: `15 < 20` và `10 < 25` → trùng. Kiểm `book(20, 30)`: `20 < 20`? Không → không trùng.

In [6]:
events = []
for start, end in [(10, 20), (15, 25), (20, 30)]:
    conflict = None
    for s, e in events:
        if start < e and s < end:
            conflict = (s, e)
            break
    print(f"book({start},{end}): xung đột với {conflict} -> {'False' if conflict else 'True'}")
    if conflict is None:
        events.append((start, end))
print("danh sách sau cùng:", events)

book(10,20): xung đột với None -> True
book(15,25): xung đột với (10, 20) -> False
book(20,30): xung đột với None -> True
danh sách sau cùng: [(10, 20), (20, 30)]


## 5. Cách 1 — Brute Force: quét mọi lịch cũ

### Ý tưởng

Lưu các sự kiện trong `list` các cặp `(s, e)`. `book(start, end)`: nếu **có** sự kiện cũ thỏa `start < e and s < end` thì trả `False`; ngược lại thêm `(start, end)` và trả `True`.

### Code

In [7]:
class CalendarBrute:
    def __init__(self):
        self.events = []

    def book(self, start: int, end: int) -> bool:
        for s, e in self.events:
            if start < e and s < end:
                return False
        self.events.append((start, end))
        return True


print(run_design(CalendarBrute, example_ops, example_args))

[None, True, False, True]


### Độ phức tạp và nút thắt

- Mỗi `book` quét tối đa mọi sự kiện đã nhận: `O(n)`; tổng cho `n` lần gọi là `O(n^2)`. Với `n <= 1000` chỉ ~`5 * 10^5` phép kiểm — vẫn rất nhanh và **đủ để qua LeetCode**.
- Bộ nhớ `O(n)`.
- Nút thắt: ta quét cả những sự kiện **ở rất xa** khoảng mới. Vì các sự kiện đã nhận không chồng nhau nên có thể **sắp xếp** chúng và chỉ nhìn một hai ứng viên gần nhất (tìm bằng nhị phân).

## 6. Ý tưởng thuật toán / bước đột phá

Các sự kiện đã nhận **đôi một rời nhau**, nên có thể xếp chúng thành dãy tăng dần theo thời gian — và khi đó thứ tự theo điểm đầu **và** thứ tự theo điểm cuối là **như nhau**:

```
[s1,e1)  [s2,e2)  [s3,e3)  ...      với  s1 < e1 <= s2 < e2 <= s3 < e3 ...
```

Khoảng mới `[start, end)` chỉ có thể chạm vào sự kiện **sát bên trái** hoặc **sát bên phải** vị trí của nó. Nghĩa là chỉ cần xét **một vài ứng viên**, tìm bằng **tìm kiếm nhị phân** (`O(log n)`), thay vì quét hết.

Có hai cách chọn ứng viên:

1. **Sắp theo điểm đầu**: ứng viên là sự kiện **đứng ngay trước** (điểm đầu `<= start`, phải có `e <= start`) và sự kiện **đứng ngay sau** (điểm đầu `> start`, phải có `s >= end`).
2. **Sắp theo điểm cuối** (cách của doocs): ứng viên **duy nhất** là sự kiện **đầu tiên có `e > start`**; trùng khi và chỉ khi điểm đầu của nó `< end`. (Các sự kiện trước nó có `e <= start` nên chắc chắn không giao; các sự kiện sau nó còn bắt đầu muộn hơn.)

Sau khi kiểm xong, chèn khoảng mới vào đúng vị trí để dãy vẫn tăng dần.

In [8]:
# Minh họa lý do "một ứng viên là đủ" khi sắp theo điểm cuối
events = [(10, 20), (30, 40), (50, 60)]             # rời nhau, đã sắp xếp
ends = [e for s, e in events]
starts = [s for s, e in events]
for start, end in [(15, 25), (22, 28), (35, 55), (5, 12), (60, 70), (0, 100)]:
    idx = bisect_right(ends, start)                 # sự kiện đầu tiên có end > start
    cand = events[idx] if idx < len(events) else None
    conflict = cand is not None and cand[0] < end
    truth = any(start < e and s < end for s, e in events)
    print(f"[{start},{end}): ứng viên = {cand}, trùng = {conflict}, đáp án thật = {truth}")
    assert conflict == truth

[15,25): ứng viên = (10, 20), trùng = True, đáp án thật = True
[22,28): ứng viên = (30, 40), trùng = False, đáp án thật = False
[35,55): ứng viên = (30, 40), trùng = True, đáp án thật = True
[5,12): ứng viên = (10, 20), trùng = True, đáp án thật = True
[60,70): ứng viên = None, trùng = False, đáp án thật = False
[0,100): ứng viên = (10, 20), trùng = True, đáp án thật = True


## 7. Các cách tiếp theo

Tổng cộng 5 cách: (1) quét mọi lịch (mục 5), (2) list đã sắp xếp theo điểm đầu + kiểm hàng xóm, (3) list sắp theo điểm cuối + một ứng viên (lời giải nộp), (4) cây nhị phân tìm kiếm, (5) segment tree động.

### 7.1. Cách 2 — List sắp theo điểm đầu + kiểm tra hai hàng xóm

Giữ hai list song song `starts` và `ends` (cùng thứ tự tăng). Với `[start, end)`:

1. `i = bisect_right(starts, start)`: số sự kiện có điểm đầu `<= start`.
2. **Hàng xóm trái** là sự kiện `i - 1` (nếu `i > 0`): trùng nếu `ends[i - 1] > start`.
3. **Hàng xóm phải** là sự kiện `i` (nếu `i < len`): trùng nếu `starts[i] < end`.
4. Nếu không trùng, chèn `start` vào `starts` và `end` vào `ends` tại vị trí `i`.

In [9]:
class CalendarNeighbors:
    def __init__(self):
        self.starts = []
        self.ends = []

    def book(self, start: int, end: int) -> bool:
        i = bisect_right(self.starts, start)
        if i > 0 and self.ends[i - 1] > start:                 # hàng xóm trái chồng lên
            return False
        if i < len(self.starts) and self.starts[i] < end:      # hàng xóm phải chồng lên
            return False
        self.starts.insert(i, start)
        self.ends.insert(i, end)
        return True


print(run_design(CalendarNeighbors, example_ops, example_args))

[None, True, False, True]


In [10]:
# Code nộp LeetCode cho cách 2
from bisect import bisect_right


class MyCalendarNeighbors:

    def __init__(self):
        self.starts = []
        self.ends = []

    def book(self, startTime: int, endTime: int) -> bool:
        i = bisect_right(self.starts, startTime)
        if i > 0 and self.ends[i - 1] > startTime:
            return False
        if i < len(self.starts) and self.starts[i] < endTime:
            return False
        self.starts.insert(i, startTime)
        self.ends.insert(i, endTime)
        return True


print(run_design(MyCalendarNeighbors, example_ops, example_args))

[None, True, False, True]


**Độ phức tạp:** tìm vị trí `O(log n)`; chèn vào list `O(n)` (dịch phần tử, nhưng là thao tác bộ nhớ rất nhanh, `n <= 1000`). Tổng cho `n` lần gọi: `O(n^2)` ở mức thao tác nhưng hằng số cực nhỏ; về so sánh chỉ `O(n log n)`. Bộ nhớ `O(n)`.

### 7.2. Cách 3 — List sắp theo điểm cuối + một ứng viên (lời giải nộp)

Bản gốc của doocs dùng `SortedDict` (khóa là `end`, giá trị là `start`). Vì chỉ dùng stdlib, ta thay bằng hai list song song `ends` (khóa, sắp tăng) và `starts` (giá trị tương ứng):

1. `i = bisect_right(ends, start)`: chỉ số của sự kiện **đầu tiên có `end' > start`**.
2. Nếu `i < len(ends)` và `starts[i] < end` → trùng → `False`.
3. Ngược lại chèn tại **đúng `i`**: vì khoảng mới không trùng nên `end <= starts[i] < ends[i]` và mọi `ends[j] <= start < end` với `j < i`; vậy `end` nằm giữa `ends[i-1]` và `ends[i]`, tức vị trí chèn chính là `i`.

In [11]:
class CalendarEndKeyed:
    def __init__(self):
        self.ends = []
        self.starts = []

    def book(self, start: int, end: int) -> bool:
        i = bisect_right(self.ends, start)            # sự kiện đầu tiên có end' > start
        if i < len(self.ends) and self.starts[i] < end:
            return False
        self.ends.insert(i, end)
        self.starts.insert(i, start)
        return True


print(run_design(CalendarEndKeyed, example_ops, example_args))

[None, True, False, True]


In [12]:
# Code nộp LeetCode cho cách 3
from bisect import bisect_right


class MyCalendarEndKeyed:

    def __init__(self):
        self.ends = []
        self.starts = []

    def book(self, startTime: int, endTime: int) -> bool:
        i = bisect_right(self.ends, startTime)
        if i < len(self.ends) and self.starts[i] < endTime:
            return False
        self.ends.insert(i, endTime)
        self.starts.insert(i, startTime)
        return True


print(run_design(MyCalendarEndKeyed, example_ops, example_args))

[None, True, False, True]


**Độ phức tạp:** như cách 2: tìm `O(log n)`, chèn `O(n)` ở mức bộ nhớ; bộ nhớ `O(n)`. Điểm hay: **chỉ một ứng viên**, nên code ngắn hơn cách 2.

### 7.3. Cách 4 — Cây nhị phân tìm kiếm (BST) các khoảng

Mỗi nút lưu một khoảng `[s, e)`. Quy ước: nằm hoàn toàn **bên trái** (`end <= nút.s`) thì đi xuống trái; hoàn toàn **bên phải** (`start >= nút.e`) thì đi xuống phải; còn lại là **trùng** → `False`. Gặp chỗ trống thì gắn nút mới.

Vì các khoảng trong cây rời nhau, mọi khoảng mới hoặc trùng với đúng nút đang xét, hoặc nằm hẳn một phía — nên định hướng như vậy là chính xác. Dùng vòng lặp (không đệ quy) để không phụ thuộc giới hạn đệ quy.

In [13]:
class _BNode:
    __slots__ = ("s", "e", "left", "right")

    def __init__(self, s, e):
        self.s, self.e = s, e
        self.left = self.right = None


class CalendarBST:
    def __init__(self):
        self.root = None

    def book(self, start: int, end: int) -> bool:
        if self.root is None:
            self.root = _BNode(start, end)
            return True
        cur = self.root
        while True:
            if end <= cur.s:                 # hoàn toàn bên trái
                if cur.left is None:
                    cur.left = _BNode(start, end)
                    return True
                cur = cur.left
            elif start >= cur.e:             # hoàn toàn bên phải
                if cur.right is None:
                    cur.right = _BNode(start, end)
                    return True
                cur = cur.right
            else:                            # chồng lên nút hiện tại
                return False

    def height(self):
        best, stack = 0, [(self.root, 1)] if self.root else []
        while stack:
            node, d = stack.pop()
            best = max(best, d)
            for ch in (node.left, node.right):
                if ch:
                    stack.append((ch, d + 1))
        return best


print(run_design(CalendarBST, example_ops, example_args))

[None, True, False, True]


In [14]:
# Code nộp LeetCode cho cách 4
class _Node:
    __slots__ = ("s", "e", "left", "right")

    def __init__(self, s, e):
        self.s, self.e = s, e
        self.left = self.right = None


class MyCalendarBST:

    def __init__(self):
        self.root = None

    def book(self, startTime: int, endTime: int) -> bool:
        if self.root is None:
            self.root = _Node(startTime, endTime)
            return True
        cur = self.root
        while True:
            if endTime <= cur.s:
                if cur.left is None:
                    cur.left = _Node(startTime, endTime)
                    return True
                cur = cur.left
            elif startTime >= cur.e:
                if cur.right is None:
                    cur.right = _Node(startTime, endTime)
                    return True
                cur = cur.right
            else:
                return False


print(run_design(MyCalendarBST, example_ops, example_args))

[None, True, False, True]


**Độ phức tạp:** mỗi `book` đi một đường từ gốc xuống: `O(h)` với `h` là chiều cao. Dữ liệu ngẫu nhiên: `h ≈ O(log n)`; nhưng nếu các lịch đến **theo thứ tự tăng dần**, cây thành một "cái que" `h = n` và mỗi `book` `O(n)` — vẫn không tệ hơn brute force, nhưng không nhanh hơn. (Muốn đảm bảo `O(log n)` cần cây tự cân bằng — không có sẵn trong stdlib.) Ta đo chiều cao ở mục 11.

### 7.4. Cách 5 — Segment tree động

Coi trục thời gian là các ô nguyên `0 .. 2^30 - 1` (`2^30 = 1073741824 > 10^9`). Mỗi nút phụ trách một đoạn `[lo, hi)` và lưu hai cờ:

- `any`: trong đoạn này **có** ít nhất một ô đã bị đặt.
- `full`: **cả đoạn** đã bị đặt (dùng khi một lịch phủ trọn đoạn của nút).

`book(start, end)` gồm hai bước:

1. **Hỏi** "trong `[start, end)` có ô nào đã đặt?" bằng `query`: nút rỗng/`any = False` → không; nút `full` hoặc đoạn của nút nằm trọn trong `[start, end)` mà `any` → có; còn lại hỏi hai con.
2. Nếu không có: **đánh dấu** `[start, end)` bằng `update` — nút nào có đoạn nằm trọn trong `[start, end)` thì `full = True` (không xuống sâu hơn), còn lại tạo con khi cần.

Chiều sâu cây là 30, nên đệ quy an toàn.

In [15]:
class _SNode:
    __slots__ = ("left", "right", "any", "full")

    def __init__(self):
        self.left = self.right = None
        self.any = self.full = False


class CalendarSegTree:
    HI = 1 << 30

    def __init__(self):
        self.root = _SNode()

    def _query(self, node, lo, hi, l, r):
        if node is None or not node.any or r <= lo or hi <= l:
            return False
        if node.full or (l <= lo and hi <= r):
            return True
        mid = (lo + hi) >> 1
        return self._query(node.left, lo, mid, l, r) or self._query(node.right, mid, hi, l, r)

    def _update(self, node, lo, hi, l, r):
        node.any = True
        if l <= lo and hi <= r:
            node.full = True
            return
        mid = (lo + hi) >> 1
        if l < mid:
            if node.left is None:
                node.left = _SNode()
            self._update(node.left, lo, mid, l, r)
        if r > mid:
            if node.right is None:
                node.right = _SNode()
            self._update(node.right, mid, hi, l, r)

    def book(self, start: int, end: int) -> bool:
        if self._query(self.root, 0, self.HI, start, end):
            return False
        self._update(self.root, 0, self.HI, start, end)
        return True


print(run_design(CalendarSegTree, example_ops, example_args))

[None, True, False, True]


In [16]:
# Code nộp LeetCode cho cách 5
class _SegNode:
    __slots__ = ("left", "right", "any", "full")

    def __init__(self):
        self.left = self.right = None
        self.any = self.full = False


class MyCalendarSegTree:

    def __init__(self):
        self.root = _SegNode()
        self.hi = 1 << 30

    def _query(self, node, lo, hi, l, r):
        if node is None or not node.any or r <= lo or hi <= l:
            return False
        if node.full or (l <= lo and hi <= r):
            return True
        mid = (lo + hi) >> 1
        return self._query(node.left, lo, mid, l, r) or self._query(node.right, mid, hi, l, r)

    def _update(self, node, lo, hi, l, r):
        node.any = True
        if l <= lo and hi <= r:
            node.full = True
            return
        mid = (lo + hi) >> 1
        if l < mid:
            if node.left is None:
                node.left = _SegNode()
            self._update(node.left, lo, mid, l, r)
        if r > mid:
            if node.right is None:
                node.right = _SegNode()
            self._update(node.right, mid, hi, l, r)

    def book(self, startTime: int, endTime: int) -> bool:
        if self._query(self.root, 0, self.hi, startTime, endTime):
            return False
        self._update(self.root, 0, self.hi, startTime, endTime)
        return True


print(run_design(MyCalendarSegTree, example_ops, example_args))

[None, True, False, True]


**Độ phức tạp:** mỗi `query`/`update` đi qua `O(log C)` nút với `C = 2^30` (tối đa ~`4 * 30` nút); bộ nhớ `O(n log C)` nút. Độc lập với `n`, nhưng hằng số lớn hơn các cách dùng `bisect`. Ưu điểm: dễ mở rộng cho các biến thể (My Calendar II/III, đếm số lần đặt chồng).

## 8. TRACE từng bước

### 8.1. Cách 3 trên ví dụ 1

Hai list `ends`, `starts` ban đầu rỗng.

| Lệnh | `i = bisect_right(ends, start)` | Ứng viên `starts[i]` | Kết luận | `ends` / `starts` sau |
|---|---|---|---|---|
| `book(10, 20)` | 0 | (không có, `i = len`) | `True` | `[20]` / `[10]` |
| `book(15, 25)` | `bisect_right([20], 15) = 0` | `starts[0] = 10 < 25` | `False` | không đổi |
| `book(20, 30)` | `bisect_right([20], 20) = 1` | (không có, `i = len`) | `True` | `[20, 30]` / `[10, 20]` |

Chú ý lệnh cuối: `bisect_right` (chứ không phải `bisect_left`) để sự kiện có `end = 20` **không** bị coi là ứng viên của `start = 20`.

### 8.2. Một chuỗi dài hơn: `(10,20), (30,40), (20,30), (5,12)`

| Lệnh | `i` | Ứng viên | Kết luận | `ends` / `starts` sau |
|---|---|---|---|---|
| `(10,20)` | 0 | không | `True` | `[20]` / `[10]` |
| `(30,40)` | `bisect_right([20], 30) = 1` | không | `True` | `[20,40]` / `[10,30]` |
| `(20,30)` | `bisect_right([20,40], 20) = 1` | `starts[1] = 30 < 30`? Không | `True` | `[20,30,40]` / `[10,20,30]` |
| `(5,12)` | `bisect_right([20,30,40], 5) = 0` | `starts[0] = 10 < 12` | `False` | không đổi |

Hàm `trace_calendar` in lại các bước này.

In [17]:
def trace_calendar(bookings):
    ends, starts = [], []
    for start, end in bookings:
        i = bisect_right(ends, start)
        if i < len(ends):
            cand = (starts[i], ends[i])
            conflict = starts[i] < end
            note = f"ứng viên [{cand[0]},{cand[1]}): {cand[0]} < {end}? {conflict}"
        else:
            conflict = False
            note = "không có ứng viên (i = len)"
        print(f"book({start},{end}): i={i}; {note} -> {'False' if conflict else 'True'}")
        if not conflict:
            ends.insert(i, end)
            starts.insert(i, start)
        print(f"    ends = {ends}, starts = {starts}")


trace_calendar([(10, 20), (15, 25), (20, 30)])
print()
trace_calendar([(10, 20), (30, 40), (20, 30), (5, 12)])

book(10,20): i=0; không có ứng viên (i = len) -> True
    ends = [20], starts = [10]
book(15,25): i=0; ứng viên [10,20): 10 < 25? True -> False
    ends = [20], starts = [10]
book(20,30): i=1; không có ứng viên (i = len) -> True
    ends = [20, 30], starts = [10, 20]

book(10,20): i=0; không có ứng viên (i = len) -> True
    ends = [20], starts = [10]
book(30,40): i=1; không có ứng viên (i = len) -> True
    ends = [20, 40], starts = [10, 30]
book(20,30): i=1; ứng viên [30,40): 30 < 30? False -> True
    ends = [20, 30, 40], starts = [10, 20, 30]
book(5,12): i=0; ứng viên [10,20): 10 < 12? True -> False
    ends = [20, 30, 40], starts = [10, 20, 30]


### 8.2b. Trace cách 2 (hàng xóm) trên chuỗi dài

Cùng chuỗi `(10,20), (30,40), (20,30), (5,12)`: cell dưới in vị trí chèn, hàng xóm trái/phải và quyết định.

In [18]:
def trace_neighbors(bookings):
    starts, ends = [], []
    for start, end in bookings:
        i = bisect_right(starts, start)
        left = (starts[i - 1], ends[i - 1]) if i > 0 else None
        right = (starts[i], ends[i]) if i < len(starts) else None
        bad_left = left is not None and left[1] > start
        bad_right = right is not None and right[0] < end
        print(f"book({start},{end}): i={i}, trái={left} chồng? {bad_left}, phải={right} chồng? {bad_right}"
              f" -> {'False' if bad_left or bad_right else 'True'}")
        if not (bad_left or bad_right):
            starts.insert(i, start)
            ends.insert(i, end)


trace_neighbors([(10, 20), (30, 40), (20, 30), (5, 12)])

book(10,20): i=0, trái=None chồng? False, phải=None chồng? False -> True
book(30,40): i=1, trái=(10, 20) chồng? False, phải=None chồng? False -> True
book(20,30): i=1, trái=(10, 20) chồng? False, phải=(30, 40) chồng? False -> True
book(5,12): i=0, trái=None chồng? False, phải=(10, 20) chồng? True -> False


### 8.3. Trace segment tree (số nút được tạo)

Với `book(10, 20)`, `book(15, 25)`, `book(20, 30)`: cell dưới đếm số nút đã tạo sau mỗi lệnh để thấy mỗi lần đặt thành công chỉ thêm khoảng `O(log C)` nút.

In [19]:
def count_nodes(node):
    if node is None:
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)


cal = CalendarSegTree()
for start, end in [(10, 20), (15, 25), (20, 30), (0, 10 ** 9), (500, 501)]:
    before = count_nodes(cal.root)
    ok = cal.book(start, end)
    print(f"book({start},{end}) -> {ok}; số nút: {before} -> {count_nodes(cal.root)}")

book(10,20) -> True; số nút: 1 -> 34
book(15,25) -> False; số nút: 34 -> 34
book(20,30) -> True; số nút: 34 -> 39
book(0,1000000000) -> False; số nút: 39 -> 39
book(500,501) -> True; số nút: 39 -> 48


## 9. Code nộp LeetCode

Chọn cách 3 (list sắp theo điểm cuối + một ứng viên): ngắn gọn, `O(log n)` cho phần tìm, đúng với doocs nhưng chỉ dùng thư viện chuẩn. Trên LeetCode, `bisect_right` có sẵn nhờ `from bisect import bisect_right`.

In [20]:
from bisect import bisect_right


class MyCalendar:

    def __init__(self):
        self.ends = []      # điểm cuối của các sự kiện, tăng dần
        self.starts = []    # starts[i] là điểm đầu của sự kiện có điểm cuối ends[i]

    def book(self, startTime: int, endTime: int) -> bool:
        i = bisect_right(self.ends, startTime)       # sự kiện đầu tiên có end > startTime
        if i < len(self.ends) and self.starts[i] < endTime:
            return False                             # nó bắt đầu trước khi sự kiện mới kết thúc
        self.ends.insert(i, endTime)
        self.starts.insert(i, startTime)
        return True


# Your MyCalendar object will be instantiated and called as such:
# obj = MyCalendar()
# param_1 = obj.book(startTime,endTime)
print(run_design(MyCalendar, example_ops, example_args))   # [None, True, False, True]

[None, True, False, True]


## 10. Vì sao đúng

**Bất biến.** Tại mọi thời điểm, các sự kiện đã nhận đôi một **rời nhau**, và `ends`, `starts` được sắp tăng theo cùng thứ tự: vì rời nhau nên với hai sự kiện `A` đứng trước `B` ta có `A.end <= B.start < B.end`, do đó `A.end < B.end` và `A.start < B.start`. Sắp theo `end` đồng thời là sắp theo `start`.

**Kiểm tra chỉ cần một ứng viên.** Gọi `i = bisect_right(ends, start)` — chỉ số đầu tiên có `ends[i] > start`.

- Với `j < i`: `ends[j] <= start`, nên sự kiện `j` kết thúc trước hoặc đúng lúc lịch mới bắt đầu → **không giao**.
- Với `j > i`: `starts[j] >= ends[i] > starts[i]`.
  - Nếu ứng viên `i` **không** giao (tức `starts[i] >= end`) thì `starts[j] >= starts[i] >= end` nên lịch mới kết thúc trước `j` bắt đầu → **không giao**.
  - Nếu ứng viên `i` giao thì đã trả `False` rồi.
- Với `j = i`: `ends[i] > start` đã biết, nên theo công thức `a < d và c < b` (với `[start, end)` và `[starts[i], ends[i])`): giao ⇔ `start < ends[i]` (đúng) và `starts[i] < end`. Vậy giao ⇔ `starts[i] < end`, đúng điều kiện trong code.

**Chèn đúng chỗ.** Khi không giao, `end <= starts[i] < ends[i]` (nếu `i` tồn tại) và `ends[i-1] <= start < end` (nếu `i > 0`), nên chèn tại `i` giữ nguyên thứ tự tăng của `ends` và (do rời nhau) cả `starts`. Bất biến được bảo toàn → quy nạp theo số lần gọi.

**Khi `False` thì không thêm:** code chỉ chèn sau khi vượt qua điều kiện, đúng yêu cầu "return false and do not add".

In [21]:
# Kiểm chứng bất biến trên một chuỗi lệnh ngẫu nhiên: ends, starts tăng ngặt, các khoảng rời nhau
rng = random.Random(1)
cal = MyCalendar()
accepted = 0
for _ in range(2000):
    s = rng.randrange(0, 5000)
    e = s + rng.randint(1, 40)
    if cal.book(s, e):
        accepted += 1
    assert all(a < b for a, b in zip(cal.ends, cal.ends[1:]))
    assert all(a < b for a, b in zip(cal.starts, cal.starts[1:]))
    assert all(cal.ends[i] <= cal.starts[i + 1] for i in range(len(cal.ends) - 1))
    assert all(cal.starts[i] < cal.ends[i] for i in range(len(cal.ends)))
print("đã nhận", accepted, "/ 2000 lịch; bất biến (rời nhau, sắp tăng) luôn đúng")

đã nhận 268 / 2000 lịch; bất biến (rời nhau, sắp tăng) luôn đúng


## 11. Time / Space Complexity

| Cách | `book` (so sánh) | Chèn/ghi | Space |
|---|---|---|---|
| 1. Quét mọi lịch | `O(n)` | `O(1)` | `O(n)` |
| 2. List theo điểm đầu + hàng xóm | `O(log n)` | `O(n)` (dịch list) | `O(n)` |
| 3. List theo điểm cuối + 1 ứng viên | `O(log n)` | `O(n)` (dịch list) | `O(n)` |
| 4. BST | `O(h)`, trung bình `O(log n)`, tệ nhất `O(n)` | `O(1)` | `O(n)` |
| 5. Segment tree động | `O(log C)`, `C = 2^30` | cùng | `O(n log C)` |

(`n` là số lịch đã nhận.) Cell dưới (a) đo chiều cao BST với dữ liệu ngẫu nhiên và dữ liệu tăng dần để thấy tệ nhất; (b) đo thời gian cách 1 so với cách 3 khi `n` lớn (vượt ràng buộc để thấy khác biệt).

In [22]:
# (a) chiều cao BST: lịch đến ngẫu nhiên vs lịch đến theo thứ tự tăng dần
n = 1000
rng = random.Random(3)
slots = list(range(n))
rng.shuffle(slots)
bst_random = CalendarBST()
for t in slots:
    assert bst_random.book(t * 10, t * 10 + 5)
bst_sorted = CalendarBST()
for t in range(n):
    assert bst_sorted.book(t * 10, t * 10 + 5)
print("BST với", n, "lịch ngẫu nhiên: cao", bst_random.height(), "(cỡ vài chục, tức O(log n))")
print("BST với", n, "lịch tăng dần  : cao", bst_sorted.height(), "(= n, suy biến thành danh sách)")

BST với 1000 lịch ngẫu nhiên: cao 24 (cỡ vài chục, tức O(log n))
BST với 1000 lịch tăng dần  : cao 1000 (= n, suy biến thành danh sách)


In [23]:
# (b) thời gian: n lịch rời nhau đến theo thứ tự ngẫu nhiên (mọi lệnh đều True)
def time_calls(cls, slots):
    t0 = time.perf_counter()
    cal = cls()
    for t in slots:
        assert cal.book(t * 10, t * 10 + 5)
    return (time.perf_counter() - t0) * 1000


for n in (1000, 4000, 8000):
    slots = list(range(n))
    random.Random(n).shuffle(slots)
    row = {name: time_calls(cls, slots) for name, cls in [("quét", CalendarBrute), ("điểm cuối", CalendarEndKeyed),
                                                              ("BST", CalendarBST), ("segtree", CalendarSegTree)]}
    print(f"n={n:5d} lệnh: " + ", ".join(f"{k} {v:8.1f} ms" for k, v in row.items()))

n= 1000 lệnh: quét     16.2 ms, điểm cuối      0.5 ms, BST      0.8 ms, segtree     10.6 ms


n= 4000 lệnh: quét    282.4 ms, điểm cuối      5.1 ms, BST      5.6 ms, segtree     69.8 ms


n= 8000 lệnh: quét   1045.4 ms, điểm cuối     16.8 ms, BST     11.7 ms, segtree    134.1 ms


## 12. Bộ test

Chiến lược: (a) ví dụ đề qua bộ chạy chuỗi lệnh; (b) edge case (sát nhau, trùng hệt, bao nhau, nằm trong nhau, biên `0` và `10^9`, đoạn dài 1); (c) phản ví dụ cho các cách sai; (d) đối chiếu **mọi cách** với một **oracle độc lập theo tập điểm** (đặt cờ từng điểm nguyên) trên tọa độ nhỏ; (e) đối chiếu mọi cách với brute trên tọa độ lớn tới `10^9`; (f) tối đa 1000 lần gọi.

In [24]:
class CalendarPoints:
    """Oracle độc lập: lưu tập các điểm nguyên đã bị chiếm (chỉ dùng với tọa độ nhỏ)."""

    def __init__(self):
        self.taken = set()

    def book(self, start, end):
        pts = set(range(start, end))
        if pts & self.taken:
            return False
        self.taken |= pts
        return True


implementations = {
    "quét": CalendarBrute,
    "hàng xóm": CalendarNeighbors,
    "điểm cuối": CalendarEndKeyed,
    "BST": CalendarBST,
    "segtree": CalendarSegTree,
    "nộp Neighbors": MyCalendarNeighbors,
    "nộp EndKeyed": MyCalendarEndKeyed,
    "nộp BST": MyCalendarBST,
    "nộp SegTree": MyCalendarSegTree,
    "nộp MyCalendar": MyCalendar,
}


def run_bookings(cls, bookings):
    cal = cls()
    return [cal.book(s, e) for s, e in bookings]


def check_all(bookings, expected=None, label=""):
    if expected is None:
        expected = run_bookings(CalendarPoints, bookings)
    for name, cls in implementations.items():
        got = run_bookings(cls, bookings)
        assert got == expected, (label, name, bookings, got, expected)


# (a) ví dụ đề qua bộ chạy chuỗi lệnh
for name, cls in implementations.items():
    assert run_design(cls, example_ops, example_args) == example_out, name
print("ví dụ đề (qua run_design) cho mọi cách: OK")

ví dụ đề (qua run_design) cho mọi cách: OK


In [25]:
# (b) edge case: nhập dưới dạng (bookings, expected)
cases = [
    ([(10, 20), (15, 25), (20, 30)], [True, False, True]),                   # ví dụ đề
    ([(10, 20), (10, 20)], [True, False]),                                   # trùng hệt
    ([(10, 20), (20, 30)], [True, True]),                                    # sát nhau
    ([(20, 30), (10, 20)], [True, True]),                                    # sát nhau, thứ tự ngược
    ([(15, 16), (10, 20)], [True, False]),                                   # lịch mới BAO lịch cũ
    ([(10, 20), (12, 15)], [True, False]),                                   # lịch mới nằm TRONG lịch cũ
    ([(10, 20), (5, 11)], [True, False]),                                    # chồng lên đầu
    ([(10, 20), (19, 25)], [True, False]),                                   # chồng 1 điểm cuối
    ([(10, 20), (5, 10)], [True, True]),                                     # sát nhau bên trái
    ([(4, 5), (6, 7), (5, 6)], [True, True, True]),                          # lấp đúng khe hở 1 đơn vị
    ([(4, 5), (6, 7), (4, 7)], [True, True, False]),                         # phủ cả khe lẫn hai bên
    ([(5, 6), (5, 6), (5, 6)], [True, False, False]),                        # đoạn dài 1 lặp lại
    ([(0, 10 ** 9)], [True]),                                                # kín cả miền
    ([(0, 10 ** 9), (0, 1)], [True, False]),
    ([(0, 10 ** 9), (10 ** 9 - 1, 10 ** 9)], [True, False]),
    ([(0, 1), (10 ** 9 - 1, 10 ** 9), (1, 10 ** 9 - 1)], [True, True, True]),  # lấp kín giữa hai đầu
    ([(0, 1), (10 ** 9 - 1, 10 ** 9), (1, 10 ** 9)], [True, True, False]),
    ([(10, 20), (30, 40), (50, 60), (25, 28), (45, 55), (60, 70)], [True, True, True, True, False, True]),
]
for bookings, expected in cases:
    check_all(bookings, expected, "edge")
print("edge case:", len(cases), "bộ: OK")

edge case: 18 bộ: OK


In [26]:
# (c) phản ví dụ: các cách sai (phân tích ở mục 13)
class WrongClosed:
    """SAI: coi hai lịch sát nhau là trùng (dùng <= thay vì <)."""
    def __init__(self):
        self.events = []

    def book(self, start, end):
        for s, e in self.events:
            if start <= e and s <= end:
                return False
        self.events.append((start, end))
        return True


class WrongStartOnly:
    """SAI: chỉ kiểm tra điểm đầu của lịch mới có nằm trong lịch cũ."""
    def __init__(self):
        self.events = []

    def book(self, start, end):
        for s, e in self.events:
            if s <= start < e:
                return False
        self.events.append((start, end))
        return True


class WrongNoStore:
    """SAI: trả True nhưng quên lưu lịch."""
    def __init__(self):
        self.events = []

    def book(self, start, end):
        for s, e in self.events:
            if start < e and s < end:
                return False
        return True


class WrongBisectLeft:
    """SAI: dùng bisect_left, coi sự kiện có end == start là ứng viên."""
    def __init__(self):
        self.ends, self.starts = [], []

    def book(self, start, end):
        i = bisect_left(self.ends, start)
        if i < len(self.ends) and self.starts[i] < end:
            return False
        self.ends.insert(i, end)
        self.starts.insert(i, start)
        return True


class WrongAlwaysStore:
    """SAI: thêm lịch cả khi trả False."""
    def __init__(self):
        self.events = []

    def book(self, start, end):
        ok = all(not (start < e and s < end) for s, e in self.events)
        self.events.append((start, end))
        return ok


def first_counterexample(cls, trials=5000, seed=11):
    rng = random.Random(seed)
    for _ in range(trials):
        bookings = []
        for _ in range(rng.randint(2, 5)):
            s = rng.randint(0, 8)
            bookings.append((s, s + rng.randint(1, 4)))
        if run_bookings(cls, bookings) != run_bookings(CalendarPoints, bookings):
            return bookings
    return None


for cls in (WrongClosed, WrongStartOnly, WrongNoStore, WrongBisectLeft, WrongAlwaysStore):
    ce = first_counterexample(cls)
    print(f"{cls.__name__:16s}: phản ví dụ {ce} -> đúng {run_bookings(CalendarPoints, ce)} | sai {run_bookings(cls, ce)}")
    assert ce is not None

WrongClosed     : phản ví dụ [(1, 2), (6, 10), (2, 3)] -> đúng [True, True, True] | sai [True, True, False]
WrongStartOnly  : phản ví dụ [(8, 12), (7, 9), (2, 6), (2, 3), (7, 10)] -> đúng [True, False, True, False, False] | sai [True, True, True, False, False]
WrongNoStore    : phản ví dụ [(8, 12), (7, 9), (2, 6), (2, 3), (7, 10)] -> đúng [True, False, True, False, False] | sai [True, True, True, True, True]
WrongBisectLeft : phản ví dụ [(1, 2), (6, 10), (2, 3)] -> đúng [True, True, True] | sai [True, True, False]
WrongAlwaysStore: phản ví dụ [(6, 10), (1, 3), (4, 7), (1, 4), (5, 6)] -> đúng [True, True, False, False, True] | sai [True, True, False, False, False]


In [27]:
# (d) tọa độ nhỏ: đối chiếu mọi cách với oracle tập điểm
rng = random.Random(729)
n_true = n_false = 0
for case in range(600):
    maxc = rng.choice([5, 10, 30, 60])
    bookings = []
    for _ in range(rng.randint(1, 60)):
        s = rng.randrange(0, maxc)
        e = rng.randint(s + 1, min(maxc, s + rng.choice([1, 3, 10, maxc])))
        bookings.append((s, e))
    expected = run_bookings(CalendarPoints, bookings)
    n_true += sum(expected)
    n_false += len(expected) - sum(expected)
    check_all(bookings, expected, f"small {case}")
print(f"600 chuỗi tọa độ nhỏ: OK (tổng {n_true} lần True, {n_false} lần False)")

# vét cạn: mọi chuỗi 3 lệnh trên miền [0, 5]
from itertools import product
iv = [(s, e) for s in range(0, 5) for e in range(s + 1, 6)]
count = 0
for trio in product(iv, repeat=3):
    check_all(list(trio), None, "exhaustive")
    count += 1
print("vét cạn", count, "chuỗi 3 lệnh trên miền [0, 5]: OK")

600 chuỗi tọa độ nhỏ: OK (tổng 4657 lần True, 13891 lần False)


vét cạn 3375 chuỗi 3 lệnh trên miền [0, 5]: OK


In [28]:
# (e) tọa độ lớn tới 10^9: đối chiếu mọi cách với brute
rng = random.Random(1000)
for case in range(200):
    bookings = []
    for _ in range(rng.randint(1, 120)):
        s = rng.randrange(0, 10 ** 9)
        e = min(10 ** 9, s + rng.choice([1, 100, 10 ** 6, 5 * 10 ** 7, 10 ** 9]))
        if e <= s:
            s, e = 10 ** 9 - 1, 10 ** 9
        bookings.append((s, e))
    expected = run_bookings(CalendarBrute, bookings)
    check_all(bookings, expected, f"large {case}")
print("200 chuỗi tọa độ lớn: OK")

# (f) ràng buộc: đúng 1000 lần gọi, với nhiều kiểu mẫu
rng = random.Random(2024)
patterns = {
    "tăng dần, rời nhau": [(t * 7, t * 7 + 3) for t in range(1000)],
    "giảm dần, rời nhau": [(t * 7, t * 7 + 3) for t in reversed(range(1000))],
    "ngẫu nhiên thưa": [(s, s + rng.randint(1, 10 ** 5)) for s in (rng.randrange(0, 10 ** 9 - 10 ** 5) for _ in range(1000))],
    "lặp lại một khoảng": [(5, 10)] * 1000,
    "lồng nhau từ ngoài vào trong": [(i, 1000 - i) for i in range(500)] + [(i, 1000 - i) for i in range(500)],
}
for label, bookings in patterns.items():
    assert len(bookings) == 1000
    expected = run_bookings(CalendarBrute, bookings)
    check_all(bookings, expected, label)
    print(f"{label:32s}: {sum(expected):4d} True / 1000")

200 chuỗi tọa độ lớn: OK
tăng dần, rời nhau              : 1000 True / 1000
giảm dần, rời nhau              : 1000 True / 1000


ngẫu nhiên thưa                 :  955 True / 1000
lặp lại một khoảng              :    1 True / 1000
lồng nhau từ ngoài vào trong    :    1 True / 1000


## 13. Các lỗi người mới hay gặp

Ta dùng `first_counterexample` (quét ngẫu nhiên nhỏ có seed, so với oracle tập điểm) để lấy **phản ví dụ thật** cho từng lỗi.

### Lỗi 1 — Dùng `<=`: coi hai lịch sát nhau là trùng

Khoảng là **nửa mở**, nên `[10, 20)` và `[20, 30)` không chung điểm nào. Viết `start <= e and s <= end` là coi khoảng đóng.

In [29]:
print("ví dụ đề: đúng", example_out[1:], "| sai", run_bookings(WrongClosed, [(10, 20), (15, 25), (20, 30)]))
ce = first_counterexample(WrongClosed)
print("phản ví dụ nhỏ:", ce, "-> đúng", run_bookings(CalendarPoints, ce), "| sai", run_bookings(WrongClosed, ce))

ví dụ đề: đúng [True, False, True] | sai [True, False, False]
phản ví dụ nhỏ: [(1, 2), (6, 10), (2, 3)] -> đúng [True, True, True] | sai [True, True, False]


**Vì sao sai:** lệnh `book(20, 30)` sau `book(10, 20)` bị từ chối dù hợp lệ. **Đúng:** giao nhau ⇔ `start < e` **và** `s < end` (bất đẳng thức **ngặt**).

### Lỗi 2 — Chỉ kiểm tra điểm đầu của lịch mới

Nghĩ rằng "trùng nghĩa là điểm đầu của tôi rơi vào lịch cũ". Thiếu trường hợp lịch mới **bao** lịch cũ.

In [30]:
bk = [(15, 16), (10, 20)]
print("đặt", bk, "-> đúng", run_bookings(CalendarPoints, bk), "| sai", run_bookings(WrongStartOnly, bk))
ce = first_counterexample(WrongStartOnly)
print("phản ví dụ tìm tự động:", ce)

đặt [(15, 16), (10, 20)] -> đúng [True, False] | sai [True, True]
phản ví dụ tìm tự động: [(8, 12), (7, 9), (2, 6), (2, 3), (7, 10)]


**Vì sao sai:** `[10, 20)` chứa `[15, 16)`, nhưng điểm đầu 10 không nằm trong `[15, 16)` nên bị bỏ sót. **Đúng:** dùng điều kiện đối xứng `start < e and s < end`, hoặc kiểm tra cả hai chiều.

### Lỗi 3 — Quên lưu lịch khi thành công

In [31]:
bk = [(10, 20), (15, 25)]
print("đặt", bk, "-> đúng", run_bookings(CalendarPoints, bk), "| sai", run_bookings(WrongNoStore, bk))

đặt [(10, 20), (15, 25)] -> đúng [True, False] | sai [True, True]


**Vì sao sai:** lịch đầu không được lưu nên lịch thứ hai được nhận dù chồng. **Đúng:** thêm `(start, end)` vào cấu trúc **trước khi** `return True`.

### Lỗi 4 — Nhầm `bisect_left` và `bisect_right` (cách sắp theo điểm cuối)

Với `bisect_left(ends, start)`, sự kiện có `end == start` (sát nhau) bị coi là ứng viên; vì `starts[i] < ends[i] = start < end` nên nó bị kết luận là trùng.

In [32]:
bk = [(10, 20), (20, 30)]
print("đặt", bk, "-> đúng", run_bookings(CalendarPoints, bk), "| sai", run_bookings(WrongBisectLeft, bk))
print("bisect_left([20], 20) =", bisect_left([20], 20), "| bisect_right([20], 20) =", bisect_right([20], 20))

đặt [(10, 20), (20, 30)] -> đúng [True, True] | sai [True, False]
bisect_left([20], 20) = 0 | bisect_right([20], 20) = 1


**Vì sao sai:** `end` của sự kiện cũ bằng `start` mới nghĩa là sát nhau, **không** chồng. Muốn "đầu tiên có `end' > start`" (ngặt) phải dùng `bisect_right`. **Đúng:** `bisect_right(ends, start)`.

### Lỗi 5 — Thêm lịch cả khi trả `False`

Thêm lịch vào danh sách ngay từ đầu rồi mới kiểm tra, hoặc quên `return` sớm.

In [33]:
bk = [(10, 20), (15, 25), (22, 30)]
print("đặt", bk, "-> đúng", run_bookings(CalendarPoints, bk), "| sai", run_bookings(WrongAlwaysStore, bk))

đặt [(10, 20), (15, 25), (22, 30)] -> đúng [True, False, True] | sai [True, False, False]


**Vì sao sai:** `(15, 25)` bị từ chối nhưng vẫn bị lưu, nên `(22, 30)` (vốn hợp lệ) bị chặn. Đề nói rõ: trả `false` thì **không thêm** sự kiện. **Đúng:** chỉ thêm sau khi chắc chắn không trùng.

## 14. Mẹo & nhận dạng pattern

- Bài "đặt lịch/ghép khoảng không chồng" → công thức `a < d and c < b` cho khoảng nửa mở. Nhớ **nửa mở**: sát nhau là hợp lệ.
- Tập khoảng rời nhau đã sắp xếp → dùng **tìm kiếm nhị phân** để chỉ xét 1–2 hàng xóm. Trong Python dùng `bisect` + `list.insert`; Java dùng `TreeMap.floorKey/ceilingKey`; C++ dùng `std::map::lower_bound`.
- Cách sắp theo **điểm cuối** cho đúng **một** ứng viên (`end > start` đầu tiên) — ít nhánh `if` hơn cách hai hàng xóm.
- Khi tọa độ cực lớn nhưng số khoảng nhỏ → **đừng** đánh dấu từng điểm; dùng danh sách khoảng, hoặc **segment tree động**.
- `n <= 1000`: quét tuyến tính `O(n^2)` tổng cũng đủ qua; cách `bisect` là phần "đẹp" để học.
- Bài thiết kế lớp: luôn tự viết **bộ chạy chuỗi lệnh** (`run_design`) và so với một oracle đơn giản để chống lỗi trạng thái.
- Hai biến thể "đường tới" quen thuộc: My Calendar II (cho phép chồng đôi, cấm chồng ba) và My Calendar III (hỏi độ chồng lớn nhất).

## 15. Biến thể

### 15.1. My Calendar II (LeetCode 731): cấm đặt chồng **ba**

Giữ hai danh sách: `booked` (mọi lịch đã nhận) và `overlaps` (các khoảng đang bị chồng hai lần). `book(s, e)`: nếu giao với một khoảng trong `overlaps` → `False`; ngược lại, với mỗi lịch trong `booked` giao với `[s, e)`, thêm phần giao vào `overlaps`; rồi thêm `[s, e)` vào `booked`.

In [34]:
class MyCalendarTwo:
    def __init__(self):
        self.booked = []
        self.overlaps = []

    def book(self, start, end):
        for s, e in self.overlaps:
            if start < e and s < end:
                return False
        for s, e in self.booked:
            if start < e and s < end:
                self.overlaps.append((max(s, start), min(e, end)))
        self.booked.append((start, end))
        return True


class CalendarTwoPoints:
    """Oracle: đếm số lần mỗi điểm bị đặt, cấm điểm nào đạt 3."""
    def __init__(self):
        self.count = {}

    def book(self, start, end):
        if any(self.count.get(x, 0) >= 2 for x in range(start, end)):
            return False
        for x in range(start, end):
            self.count[x] = self.count.get(x, 0) + 1
        return True


print(run_design(MyCalendarTwo, ["MyCalendarTwo"] + ["book"] * 6,
                 [[], [10, 20], [50, 60], [10, 40], [5, 15], [5, 10], [25, 55]]))
# [None, True, True, True, False, True, True]
rng = random.Random(731)
for _ in range(300):
    bk = []
    for _ in range(rng.randint(1, 25)):
        s = rng.randrange(0, 20)
        bk.append((s, rng.randint(s + 1, 20)))
    assert run_bookings(MyCalendarTwo, bk) == run_bookings(CalendarTwoPoints, bk)
print("My Calendar II khớp oracle điểm: OK")

[None, True, True, True, False, True, True]
My Calendar II khớp oracle điểm: OK


### 15.2. My Calendar III (LeetCode 732): độ chồng lớn nhất sau mỗi lần đặt

Dùng mảng hiệu (difference map): `+1` tại `start`, `-1` tại `end`; quét theo thứ tự khóa để tính độ chồng lớn nhất.

In [35]:
class MyCalendarThree:
    def __init__(self):
        self.diff = {}

    def book(self, startTime, endTime):
        self.diff[startTime] = self.diff.get(startTime, 0) + 1
        self.diff[endTime] = self.diff.get(endTime, 0) - 1
        cur = best = 0
        for t in sorted(self.diff):
            cur += self.diff[t]
            best = max(best, cur)
        return best


print(run_design(MyCalendarThree, ["MyCalendarThree"] + ["book"] * 6,
                 [[], [10, 20], [50, 60], [10, 40], [5, 15], [5, 10], [25, 55]]))
# [None, 1, 1, 2, 3, 3, 3]

[None, 1, 1, 2, 3, 3, 3]


### 15.3. Thêm thao tác huỷ sự kiện (`cancel`)

Với cách 3, huỷ một sự kiện đã có là tìm đúng chỉ số bằng `bisect_left(ends, end)` rồi `del` ở cả hai list.

In [36]:
class CancellableCalendar(CalendarEndKeyed):
    def cancel(self, start, end):
        i = bisect_left(self.ends, end)
        if i < len(self.ends) and self.ends[i] == end and self.starts[i] == start:
            del self.ends[i]
            del self.starts[i]
            return True
        return False


print(run_design(CancellableCalendar, ["CancellableCalendar", "book", "book", "cancel", "book", "cancel"],
                 [[], [10, 20], [15, 25], [10, 20], [15, 25], [1, 2]]))
# [None, True, False, True, True, False]

[None, True, False, True, True, False]


### 15.4. Tìm khoảng trống: thời điểm sớm nhất đặt được sự kiện dài `L`

Duyệt các khoảng đã sắp xếp theo thứ tự và xét các "khe" giữa chúng (đầu miền → sự kiện đầu, giữa hai sự kiện liên tiếp, sau sự kiện cuối).

In [37]:
def earliest_slot(cal, length, horizon=10 ** 9):
    """Thời điểm bắt đầu sớm nhất để đặt được sự kiện dài `length` (không vượt `horizon`)."""
    prev_end = 0
    for s, e in zip(cal.starts, cal.ends):
        if s - prev_end >= length:
            return prev_end
        prev_end = e
    return prev_end if horizon - prev_end >= length else None


cal = MyCalendar()
for s, e in [(10, 20), (30, 40), (22, 25)]:
    cal.book(s, e)
print([ (s, e) for s, e in zip(cal.starts, cal.ends) ])
print(earliest_slot(cal, 5), earliest_slot(cal, 10), earliest_slot(cal, 11), earliest_slot(cal, 10 ** 9))

[(10, 20), (22, 25), (30, 40)]
0 0 40 None


## 16. So sánh các cách

| Cách | Ý tưởng | Time `book` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét mọi lịch | Thử từng lịch cũ với `start < e and s < end` | `O(n)` | `O(n)` | `n <= 1000` vẫn đủ; dễ viết, oracle kiểm tra |
| 2. Sắp theo điểm đầu + hàng xóm | `bisect_right(starts, start)`, xét lịch trái/phải | `O(log n)` + chèn | `O(n)` | Khi muốn hiểu rõ quan hệ hàng xóm |
| 3. Sắp theo điểm cuối + 1 ứng viên | `bisect_right(ends, start)` rồi so `starts[i] < end` | `O(log n)` + chèn | `O(n)` | **Lời giải nộp**, ngắn nhất |
| 4. BST khoảng | Đi trái/phải theo vị trí, gặp chồng thì dừng | `O(h)` | `O(n)` | Dữ liệu ngẫu nhiên; minh họa cấu trúc cây |
| 5. Segment tree động | Hỏi/đánh dấu trên trục `[0, 2^30)` | `O(log C)` | `O(n log C)` | Cần mở rộng (My Calendar II/III, đếm chồng) |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(overlap(1, 5, 5, 9))
print(bisect_right([10, 20, 30], 20), bisect_left([10, 20, 30], 20))
print(run_bookings(MyCalendar, [(1, 5), (5, 9), (4, 6)]))
```

In [38]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Chạy cách 3 bằng tay cho chuỗi `(5, 10), (20, 30), (10, 20), (8, 12), (30, 31)`. Với mỗi lệnh ghi `i = bisect_right(ends, start)`, kết quả `True/False` và hai list `ends`, `starts` sau lệnh.

In [39]:
# Bài 2: điền kết quả tính tay
hand_results = None        # list 5 giá trị True/False
hand_ends = None           # ends sau lệnh cuối
hand_starts = None         # starts sau lệnh cuối

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành lớp theo cách 3 (sắp theo điểm cuối).

In [40]:
class ExerciseCalendar3:
    def __init__(self):
        # TODO 1: hai list rỗng ends và starts
        pass

    def book(self, start: int, end: int) -> bool:
        # TODO 2: i = vị trí đầu tiên có end' > start (dùng bisect_right)
        # TODO 3: nếu i còn trong phạm vi và starts[i] < end thì trả False
        # TODO 4: chèn end vào ends và start vào starts tại vị trí i, rồi trả True
        return None


# assert run_design(ExerciseCalendar3, ["ExerciseCalendar3", "book", "book", "book"], [[], [10, 20], [15, 25], [20, 30]]) == [None, True, False, True]
# assert run_bookings(ExerciseCalendar3, [(5, 6), (5, 6), (6, 7)]) == [True, False, True]

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 5, hãy tự viết `ExerciseTreeCalendar` bằng **BST** (cách 4): đi trái nếu `end <= node.s`, đi phải nếu `start >= node.e`, ngược lại trả `False`.

In [41]:
class ExerciseTreeCalendar:
    def __init__(self):
        pass

    def book(self, start: int, end: int) -> bool:
        # TODO
        return None


# assert run_bookings(ExerciseTreeCalendar, [(10, 20), (15, 25), (20, 30)]) == [True, False, True]
# assert run_bookings(ExerciseTreeCalendar, [(15, 16), (10, 20)]) == [True, False]

### Bài 5 — Mini challenge: My Calendar với sức chứa `K` (khó)

Cho phép **tối đa `K` sự kiện chồng nhau tại mỗi thời điểm** (bài gốc là `K = 1`; My Calendar II là `K = 2`). Viết lớp `ExerciseCapacityCalendar(K)` với `book(start, end)` trả `True` nếu sau khi thêm, không thời điểm nào có hơn `K` sự kiện. Gợi ý: dùng mảng hiệu trên các tọa độ đã gặp, quét tìm độ chồng lớn nhất trong `[start, end)` **sau khi** thử thêm.

```python
cal = ExerciseCapacityCalendar(2)
[cal.book(10, 20), cal.book(15, 25), cal.book(12, 18)] -> [True, True, False]
```

In [42]:
class ExerciseCapacityCalendar:
    def __init__(self, K: int):
        self.K = K
        # TODO: lưu danh sách sự kiện

    def book(self, start: int, end: int) -> bool:
        # TODO
        return None


# cal = ExerciseCapacityCalendar(2)
# assert [cal.book(10, 20), cal.book(15, 25), cal.book(12, 18)] == [True, True, False]
# cal1 = ExerciseCapacityCalendar(1)
# assert [cal1.book(10, 20), cal1.book(20, 30), cal1.book(19, 21)] == [True, True, False]

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: Khoảng nửa mở: `[1,5)` và `[5,9)` có chung điểm không?
- Mức 2: `bisect_right` đặt sau các số bằng, `bisect_left` đặt trước.
- Mức 3: `False`; `2 1`; `[True, True, False]`.

**Bài 2**
- Mức 1: Làm lần lượt từng lệnh, ghi cả hai list sau mỗi lệnh.
- Mức 2: `(8,12)` sau khi đã có `[5,10)`: `bisect_right([10,20,30], 8) = 0`, `starts[0] = 5 < 12`.
- Mức 3: `[True, True, True, False, True]`; `ends = [10, 20, 30, 31]`, `starts = [5, 10, 20, 30]`.

**Bài 3**
- Mức 1: `self.ends = []`, `self.starts = []`.
- Mức 2: `i = bisect_right(self.ends, start)`.
- Mức 3: Xem lời giải ở mục 9.

**Bài 4**
- Mức 1: Mỗi nút có `s, e, left, right`.
- Mức 2: Vòng `while True`; gặp `None` thì gắn nút mới và trả `True`.
- Mức 3: Xem `CalendarBST` ở mục 7.3.

**Bài 5**
- Mức 1: Độ chồng tại thời điểm `t` = số sự kiện `s <= t < e`.
- Mức 2: Độ chồng chỉ thay đổi tại các điểm đầu; chỉ cần kiểm tra tại `start` và các điểm đầu nằm trong `[start, end)`.
- Mức 3: Xem đáp án mục 19.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [43]:
print(overlap(1, 5, 5, 9))                                   # False
print(bisect_right([10, 20, 30], 20), bisect_left([10, 20, 30], 20))   # 2 1
print(run_bookings(MyCalendar, [(1, 5), (5, 9), (4, 6)]))    # [True, True, False]

assert overlap(1, 5, 5, 9) is False
assert (bisect_right([10, 20, 30], 20), bisect_left([10, 20, 30], 20)) == (2, 1)
assert run_bookings(MyCalendar, [(1, 5), (5, 9), (4, 6)]) == [True, True, False]

False
2 1
[True, True, False]


### Đáp án Bài 2

Chuỗi `(5,10), (20,30), (10,20), (8,12), (30,31)`:

| Lệnh | `i` | Kết quả | `ends` / `starts` sau |
|---|---|---|---|
| `(5,10)` | 0 | `True` | `[10]` / `[5]` |
| `(20,30)` | `bisect_right([10], 20) = 1` | `True` | `[10,30]` / `[5,20]` |
| `(10,20)` | `bisect_right([10,30], 10) = 1`, `starts[1] = 20 < 20`? Không | `True` | `[10,20,30]` / `[5,10,20]` |
| `(8,12)` | `bisect_right([10,20,30], 8) = 0`, `starts[0] = 5 < 12` | `False` | không đổi |
| `(30,31)` | `bisect_right([10,20,30], 30) = 3` = `len` | `True` | `[10,20,30,31]` / `[5,10,20,30]` |

In [44]:
hand_results = [True, True, True, False, True]
hand_ends = [10, 20, 30, 31]
hand_starts = [5, 10, 20, 30]

cal = MyCalendar()
got = [cal.book(s, e) for s, e in [(5, 10), (20, 30), (10, 20), (8, 12), (30, 31)]]
print(got, cal.ends, cal.starts)
assert got == hand_results
assert cal.ends == hand_ends and cal.starts == hand_starts
print("Bài 2 OK")

[True, True, True, False, True] [10, 20, 30, 31] [5, 10, 20, 30]
Bài 2 OK


### Đáp án Bài 3

In [45]:
class ExerciseCalendar3Solution:
    def __init__(self):
        self.ends = []
        self.starts = []

    def book(self, start: int, end: int) -> bool:
        i = bisect_right(self.ends, start)
        if i < len(self.ends) and self.starts[i] < end:
            return False
        self.ends.insert(i, end)
        self.starts.insert(i, start)
        return True


assert run_design(ExerciseCalendar3Solution, ["ExerciseCalendar3Solution", "book", "book", "book"],
                  [[], [10, 20], [15, 25], [20, 30]]) == [None, True, False, True]
assert run_bookings(ExerciseCalendar3Solution, [(5, 6), (5, 6), (6, 7)]) == [True, False, True]
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [46]:
class ExerciseTreeCalendarSolution:
    class _N:
        __slots__ = ("s", "e", "l", "r")

        def __init__(self, s, e):
            self.s, self.e, self.l, self.r = s, e, None, None

    def __init__(self):
        self.root = None

    def book(self, start: int, end: int) -> bool:
        node = self._N(start, end)
        if self.root is None:
            self.root = node
            return True
        cur = self.root
        while True:
            if end <= cur.s:
                if cur.l is None:
                    cur.l = node
                    return True
                cur = cur.l
            elif start >= cur.e:
                if cur.r is None:
                    cur.r = node
                    return True
                cur = cur.r
            else:
                return False


assert run_bookings(ExerciseTreeCalendarSolution, [(10, 20), (15, 25), (20, 30)]) == [True, False, True]
assert run_bookings(ExerciseTreeCalendarSolution, [(15, 16), (10, 20)]) == [True, False]
rng = random.Random(4)
for _ in range(200):
    bk = []
    for _ in range(rng.randint(1, 30)):
        s = rng.randrange(0, 40)
        bk.append((s, rng.randint(s + 1, 40)))
    assert run_bookings(ExerciseTreeCalendarSolution, bk) == run_bookings(CalendarPoints, bk)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

Sau khi thử thêm `[start, end)`, độ chồng lớn nhất trong `[start, end)` đạt giá trị lớn nhất tại `start` hoặc tại một điểm đầu của sự kiện nằm trong khoảng đó. Ta đếm số sự kiện (kể cả sự kiện mới) bao phủ từng điểm ứng viên.

In [47]:
class ExerciseCapacityCalendarSolution:
    def __init__(self, K: int):
        self.K = K
        self.events = []

    def _depth(self, t, extra):
        return sum(1 for s, e in self.events + [extra] if s <= t < e)

    def book(self, start: int, end: int) -> bool:
        new = (start, end)
        candidates = [start] + [s for s, e in self.events if start < s < end]
        if any(self._depth(t, new) > self.K for t in candidates):
            return False
        self.events.append(new)
        return True


class CapacityOracle:
    """Oracle theo tập điểm."""
    def __init__(self, K):
        self.K, self.count = K, {}

    def book(self, start, end):
        if any(self.count.get(x, 0) >= self.K for x in range(start, end)):
            return False
        for x in range(start, end):
            self.count[x] = self.count.get(x, 0) + 1
        return True


cal = ExerciseCapacityCalendarSolution(2)
assert [cal.book(10, 20), cal.book(15, 25), cal.book(12, 18)] == [True, True, False]
cal1 = ExerciseCapacityCalendarSolution(1)
assert [cal1.book(10, 20), cal1.book(20, 30), cal1.book(19, 21)] == [True, True, False]

rng = random.Random(5)
for K in (1, 2, 3):
    for _ in range(200):
        a = ExerciseCapacityCalendarSolution(K)
        b = CapacityOracle(K)
        for _ in range(rng.randint(1, 25)):
            s = rng.randrange(0, 25)
            e = rng.randint(s + 1, 25)
            assert a.book(s, e) == b.book(s, e), (K, s, e)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
from bisect import bisect_right


class MyCalendar:

    def __init__(self):
        self.ends = []      # tăng dần
        self.starts = []    # starts[i] ứng với ends[i]

    def book(self, startTime: int, endTime: int) -> bool:
        i = bisect_right(self.ends, startTime)         # sự kiện đầu tiên có end > startTime
        if i < len(self.ends) and self.starts[i] < endTime:
            return False                               # bị chồng
        self.ends.insert(i, endTime)
        self.starts.insert(i, startTime)
        return True
```

Công thức cần nhớ: khoảng nửa mở `[a, b)` và `[c, d)` giao nhau ⇔ `a < d và c < b`; sát nhau (`b == c`) là **không** giao.

### Checklist trước khi submit

- [ ] Dùng bất đẳng thức **ngặt** (khoảng nửa mở): `[10,20)` và `[20,30)` đặt được cùng nhau.
- [ ] Chọn đúng `bisect_right` (không phải `bisect_left`) khi tìm "đầu tiên có `end' > start`".
- [ ] Chỉ **lưu** sự kiện khi trả `True`; khi `False` thì không thêm.
- [ ] Xét cả trường hợp lịch mới **bao** hoặc **nằm trong** lịch cũ (công thức hai chiều bao quát cả hai).
- [ ] Đã thử biên: `(0, 10^9)`, đoạn dài 1, lặp lại cùng một lịch, đặt theo thứ tự tăng dần/giảm dần.
- [ ] Đúng khung LeetCode: `class MyCalendar`, `__init__(self)`, `book(self, startTime: int, endTime: int) -> bool`.